In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Install and Imports

In [ ]:
!pip install sentencepiece
!pip install transformers

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration, T5Config, AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.optimization import AdamW
from tqdm import tqdm

## Model

In [ ]:
# Define the dataset class
class KeyTextDataset(Dataset):
    def __init__(self, keys, texts, tokenizer):
        self.keys = keys
        self.texts = texts
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]
        text = self.texts[idx]
        key_encoding = self.tokenizer(
            key,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
        )

        text_encoding = self.tokenizer(
            text,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
        )
        input_ids = key_encoding['input_ids'].squeeze()
        attention_mask = key_encoding['attention_mask'].squeeze()

        # print(text_encoding)

        labels = text_encoding['input_ids'].squeeze()
        labels[labels == 0] = -100
        labels_attention_mask = text_encoding['attention_mask'].squeeze()

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'labels_attention_mask':labels_attention_mask,
            'text': text
        }

# Function to train the model
def train_model(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0

    # progress_bar = tqdm(enumerate(dataloader), total=len(dataloader))
    for step, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        labels_attention_mask = batch['labels_attention_mask'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_attention_mask=labels_attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        # progress_bar.set_description(f"Train Loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

# Function to validate the model
def validate_model(model, dataloader, device):
    model.eval()
    total_loss = 0

    # progress_bar = tqdm(enumerate(dataloader), total=len(dataloader))
    for step, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        labels_attention_mask = batch['labels_attention_mask'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_attention_mask=labels_attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        # progress_bar.set_description(f"Train Loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

# Function to save the trained model and tokenizer
def save_model(model, tokenizer, output_dir):
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Model and tokenizer saved to '{output_dir}'")

# Function to load the saved model and tokenizer
def load_model(output_dir):
    model = AutoModelForSeq2SeqLM.from_pretrained(output_dir)
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    print(f"Model and tokenizer loaded from '{output_dir}'")
    return model, tokenizer

# Function to generate text given a key
def generate_text(key):
    input_ids = loaded_tokenizer.encode(key, return_tensors='pt',add_special_tokens=True).to(device)

    with torch.no_grad():
      outputs = loaded_model.generate(
          input_ids=input_ids,
          max_length =64,
          num_beams =2,
          early_stopping =True,
          num_return_sequences = 1,
          repetition_penalty= 2.5,
          length_penalty= 1.0)

    # print(outputs)

    preds = [loaded_tokenizer.decode(g,skip_special_tokens=True,clean_up_tokenization_spaces=True) for g in outputs]

    generated_text = preds[0]
    return generated_text

def predict(key):
  return generate_text(key)

In [ ]:
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_dir = "/content/drive/MyDrive/BengaliKey2Text/ModelV10V3"
# Initialize the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)
model.to(device)

## Dataset Load

In [ ]:
import pandas as pd
df =  pd.read_csv('/content/drive/MyDrive/BengaliKey2Text/DataV3/strkeysDfV01.csv')
df.columns = ["keywords", "text"]
# df = df.iloc[1579900:2000000]
# df = df.iloc[1579900:1679900]
df = df.iloc[2100000:3000000]
df = df.sample(n=10000)
df = df.reset_index(drop=True)
# df = df.head(500)
df

,keywords,text
0,প্রায়ই প্রতীক হতে একটা পেরিয়ে হয়—পঞ্চাশ মুখোমু...,বলিউডের তিন খানকে প্রায়ই একটা প্রশ্নের মুখোমুখ...
1,স্থানে বা কোনো বোঝা অন্য পড়ার অন্য দুটোই কোচিং...,নাকি অন্য উপায়ে বইয়ের বোঝা ও পড়ার বোঝা দুটোই স...
2,আমার আত্মা দেখেই শুকিয়ে,দেখেই আমার আত্মা শুকিয়ে গেল।
3,বড় অবহেলিত দুই হয়েছে নির্বাচন ঐক্যের বক্তৃতায় ...,"উপজেলা নির্বাচন প্রসঙ্গে তিনি বলেন, এটি নির্দল..."
4,বলেন কথা সেই সময়ের,সেই সময়ের কিছু কথা বলেন।
...,...,...
9995,অনুসরণ মুসলিমদের সময় যেসব নিপীড়ন সরকার রাশিয়া ...,ভারতীয় রাষ্ট্রকে লক্ষ্যবস্তু বলে চিহ্নিত করার ...
9996,অভিযোগ করেনি ব্যাপারে পর্যন্ত গতকাল তবে,তবে গতকাল পর্যন্ত এ ব্যাপারে থানায় কেউ অভিযোগ ...
9997,জিন্দেগি গেছে কিং কসৌটি টুতে দেখা জানা কে ভাষ্...,"জানা গেছে, বলিউডের কিং খানকে ‘কসৌটি জিন্দেগি ক..."
9998,সম্ভবত অনুশীলন থেকেই বেল অভিমান উঠেছিল,"তখনই গুঞ্জন উঠেছিল, সম্ভবত অভিমান থেকেই বেল অন..."


In [ ]:
print(df.isnull().sum())

keywords    0
text        0
dtype: int64


In [ ]:
df.dropna(inplace=True)

In [ ]:
print(df.isnull().sum())

keywords    0
text        0
dtype: int64


In [ ]:
# Load your dataset
keys = df['keywords'].tolist()  # List of keys
texts = df['text'].tolist()  # List of corresponding texts

In [ ]:
type(texts)

list

In [ ]:
df.to_csv('/content/drive/MyDrive/BengaliKey2Text/DataV3/toPrediction.csv')

## Prediction

In [ ]:
import pandas as pd
df =  pd.read_csv('/content/drive/MyDrive/BengaliKey2Text/toPrediction.csv')
df = df.head(1000)
# df = df.iloc[2100000:3000000]

In [ ]:
loading_model_dir = "/content/drive/MyDrive/BengaliKey2Text/ModelV10V3"
loaded_model, loaded_tokenizer = load_model(loading_model_dir)
loaded_model.to(device)

In [ ]:
key = "বই পড়"
predict(key)

'বই পড় না।'

In [ ]:
key = "বই পড়"
predict(key)

'বই পড় না।'

In [ ]:
key = "কেমন ডাটাসেট সময় ভাই বানাতে"
predict(key)

'ভাই, ডাটাসেট বানাতে কেমন সময় লাগে?'

In [ ]:
key = "কেমন ডাটাসেট সময় বানাতে"
predict(key)

'ডাটাসেট বানাতে কেমন সময় লাগে?'

In [ ]:
key = "ঠিকমতো এই অভাবের তাই শিক্ষার্থীরা কারণে"
predict(key)

'তাই এই অভাবের কারণে শিক্ষার্থীরা ঠিকমতো পড়তে পারছে না।'

In [ ]:
key = "ঠিকমতো এই অভাবের তাই শিক্ষার্থীরা কারণে"
predict(key)

'তাই এই অভাবের কারণে শিক্ষার্থীরা ঠিকমতো পড়তে পারছে না।'

In [ ]:
keywords = "দেখা হয়নি মনে হয় ছোটবেলার স্মৃতি"
predict(keywords)

'ছোটবেলার স্মৃতি মনে হয় দেখা হয়নি তাঁর।'

In [ ]:
genDf = pd.DataFrame(columns = ["keywords", "text", "generatedText"])

rows = df.shape[0]

for i in range(rows):
  keywords = df['keywords'][i]
  generatedText = predict(keywords)

  genDf.loc[i] = [df['keywords'][i], df['text'][i], generatedText]
  print(i, '  ', df['keywords'][i], '\n', df['text'][i], '\n', generatedText)

In [ ]:
genDf.to_csv('/content/drive/MyDrive/BengaliKey2Text/predictedData1.csv')

In [ ]:
import pandas as pd

url = '/content/drive/MyDrive/BengaliKey2Text/DataV3/predictedData'

df1 = pd.read_csv(url + "1.csv")
df2 = pd.read_csv(url + "2.csv")
df3 = pd.read_csv(url + "3.csv")
df4 = pd.read_csv(url + "4.csv")
df5 = pd.read_csv(url + "5.csv")
df6 = pd.read_csv(url + "6.csv")

frames = [df1, df2, df3, df4, df5, df6]
genDfCombined = pd.concat(frames)
genDfCombined.to_csv(url + ".csv", index=False)

## Get Score

### Rouge

In [ ]:
import sys
print(sys.getrecursionlimit())
sys.setrecursionlimit(8200)
print(sys.getrecursionlimit())

!pip install rouge

from rouge import Rouge


def rougeScore(test_df, columnName1, columnName2):

    newDf = test_df.filter([columnName1, columnName2], axis=1)

    newDf =  newDf.reset_index()

    lenS = newDf.shape[0]

    rscore1r = 0
    rscore1p = 0
    rscore1f = 0
    rscore2r = 0
    rscore2p = 0
    rscore2f = 0
    rscorelr = 0
    rscorelp = 0
    rscorelf = 0
    rouge = Rouge()

    for i in range(lenS):

        score = rouge.get_scores(newDf[columnName1][i], newDf[columnName2][i])

        rscore1r = rscore1r + score[0]['rouge-1']['r']
        rscore1p = rscore1p + score[0]['rouge-1']['p']
        rscore1f = rscore1f + score[0]['rouge-1']['f']

        rscore2r = rscore2r + score[0]['rouge-2']['r']
        rscore2p = rscore2p + score[0]['rouge-2']['p']
        rscore2f = rscore2f + score[0]['rouge-2']['f']

        rscorelr = rscorelr + score[0]['rouge-l']['r']
        rscorelp = rscorelp + score[0]['rouge-l']['p']
        rscorelf = rscorelf + score[0]['rouge-l']['f']

        rscore = {'rouge-1': {'r': rscore1r/lenS,
                            'p': rscore1p/lenS,
                            'f': rscore1f/lenS},
                  'rouge-2': {'r': rscore2r/lenS,
                            'p': rscore2p/lenS,
                            'f': rscore2f/lenS},
                  'rouge-l': {'r': rscorelr/lenS,
                            'p': rscorelp/lenS,
                            'f': rscorelf/lenS}}

    return rscore

8200
8200
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


### BLEU

In [ ]:
import nltk

def bleuCorpass(df, columnName1, columnName2):
    ble1score = 0
    ble2score = 0
    ble3score = 0
    ble4score = 0
    lenS = df.shape[0]
    for i in range(lenS):
        hypothesis = df[columnName2][i].split()
        reference = df[columnName1][i].split()
        references = [reference]
        list_of_references = [references]
        list_of_hypotheses = [hypothesis]
        ble1score = ble1score + nltk.translate.bleu_score.corpus_bleu(list_of_references, list_of_hypotheses, weights=(1, 0, 0, 0))
        ble2score = ble2score + nltk.translate.bleu_score.corpus_bleu(list_of_references, list_of_hypotheses, weights=(0.5, 0.5, 0, 0))
        ble3score = ble3score + nltk.translate.bleu_score.corpus_bleu(list_of_references, list_of_hypotheses, weights=(0.33, 0.33, 0.33, 0))
        ble4score = ble4score + nltk.translate.bleu_score.corpus_bleu(list_of_references, list_of_hypotheses, weights=(0.25, 0.25, 0.25, 0.25))

    bscore = {'Bleu 1': ble1score/lenS,
              'Bleu 2': ble2score/lenS,
              'Bleu 3': ble3score/lenS,
              'Bleu 4': ble4score/lenS}

    return bscore

### BERT Score

In [ ]:
import sys
sys.setrecursionlimit(8200)

!pip install evaluate
!pip install bert_score
from evaluate import load
bertscore = load("bertscore")

def bertScore(df, columnName1, columnName2):
    df = df.filter([columnName1, columnName2], axis=1)

    lenS = df.shape[0]
    bert_precision = 0
    bert_recall = 0
    bert_f1 = 0

    for i in range(lenS):
        results = bertscore.compute(predictions=[df[columnName2][i]], references=[df[columnName1][i]], lang="bn")

        bert_precision = bert_precision + results['precision'][0]
        bert_recall = bert_recall + results['recall'][0]
        bert_f1 = bert_f1 + results['f1'][0]

    bert_score = {'precision': bert_precision/lenS,
                    'recall': bert_recall/lenS,
                    'f1': bert_f1/lenS}
    return bert_score

### Meteor WIL WER

In [ ]:
from collections import Counter

class MeteorScore:
    def __init__(self, alpha=0.5, beta=0.5, gamma=0.5):
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

    def preprocess_sentence(self, sentence):
        words = sentence.split()
        return words

    def ngram_count(self, sentence, n):
        words = self.preprocess_sentence(sentence)
        ngrams = [tuple(words[i:i+n]) for i in range(len(words)-n+1)]
        return Counter(ngrams)

    def compute_precision(self, hypothesis, reference, n):
        hyp_counts = self.ngram_count(hypothesis, n)
        ref_counts = self.ngram_count(reference, n)
        overlap = sum((hyp_counts & ref_counts).values())
        precision = overlap / sum(hyp_counts.values()) if sum(hyp_counts.values()) > 0 else 0
        return precision

    def compute_recall(self, hypothesis, reference, n):
        hyp_counts = self.ngram_count(hypothesis, n)
        ref_counts = self.ngram_count(reference, n)
        overlap = sum((hyp_counts & ref_counts).values())
        recall = overlap / sum(ref_counts.values()) if sum(ref_counts.values()) > 0 else 0
        return recall

    def meteor_score(self, hypothesis, reference):
        precision = self.alpha * self.compute_precision(hypothesis, reference, 1) + (1-self.alpha) * self.compute_precision(hypothesis, reference, 2)
        recall = self.beta * self.compute_recall(hypothesis, reference, 1) + (1-self.beta) * self.compute_recall(hypothesis, reference, 2)
        fmean = (1-self.gamma) * precision + self.gamma * recall if (precision != 0 and recall != 0) else 0
        return fmean

In [ ]:
!pip install jiwer
import jiwer

meteor = MeteorScore(alpha=0.5, beta=0.5, gamma=0.5)

def wilWerMeteorScore(df, columnName1, columnName2):
    df = df.filter([columnName1, columnName2], axis=1)

    lenS = df.shape[0]
    meteor_score = 0
    wer_score = 0
    wil_score = 0

    for i in range(lenS):
      meteor_result = meteor.meteor_score(df[columnName2][i], df[columnName1][i])
      wer_result = jiwer.wer(df[columnName1][i], df[columnName2][i])
      wil_result = jiwer.wil(df[columnName1][i], df[columnName2][i])

      meteor_score = meteor_score + meteor_result
      wer_score = wer_score + wer_result
      wil_score = wil_score + wil_result

    wilWerMeteor_score = {'METEOR': meteor_score/lenS,
                    'WER': wer_score/lenS,
                    'WIL': wil_score/lenS}

    return wilWerMeteor_score

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 25.5 MB/s eta 0:00:00


### Scores

In [ ]:
import pandas as pd
df =  pd.read_csv('/content/drive/MyDrive/BengaliKey2Text/DataV3/predictedData.csv')

In [ ]:
df

,keywords,text,generatedText
0,প্রায়ই প্রতীক হতে একটা পেরিয়ে হয়—পঞ্চাশ মুখোমু...,বলিউডের তিন খানকে প্রায়ই একটা প্রশ্নের মুখোমুখ...,"আজও এর মুখোমুখি হতে হয়—পঞ্চাশ পেরিয়ে গেল, প্..."
1,স্থানে বা কোনো বোঝা অন্য পড়ার অন্য দুটোই কোচিং...,নাকি অন্য উপায়ে বইয়ের বোঝা ও পড়ার বোঝা দুটোই স...,অন্য কোনো স্থানে পড়ার টেবিলে শিশুদের বই ও কম্...
2,আমার আত্মা দেখেই শুকিয়ে,দেখেই আমার আত্মা শুকিয়ে গেল।,আমার আত্মা শুকিয়ে গেছে দেখেই।
3,বড় অবহেলিত দুই হয়েছে নির্বাচন ঐক্যের বক্তৃতায় ...,"উপজেলা নির্বাচন প্রসঙ্গে তিনি বলেন, এটি নির্দল...","তিনি বলেন, এটি দেশের সংবিধানকেও অবহেলিত করে।সভ..."
4,বলেন কথা সেই সময়ের,সেই সময়ের কিছু কথা বলেন।,তিনি সেই সময়ের কথা বলেন।
...,...,...,...
9995,অনুসরণ মুসলিমদের সময় যেসব নিপীড়ন সরকার রাশিয়া ...,ভারতীয় রাষ্ট্রকে লক্ষ্যবস্তু বলে চিহ্নিত করার ...,কাশ্মীরে মুসলিমদের ওপর যেসব নিপীড়ন চালানো হয়...
9996,অভিযোগ করেনি ব্যাপারে পর্যন্ত গতকাল তবে,তবে গতকাল পর্যন্ত এ ব্যাপারে থানায় কেউ অভিযোগ ...,তবে এ ব্যাপারে গতকাল পর্যন্ত কেউ অভিযোগ করেনি।
9997,জিন্দেগি গেছে কিং কসৌটি টুতে দেখা জানা কে ভাষ্...,"জানা গেছে, বলিউডের কিং খানকে ‘কসৌটি জিন্দেগি ক...","ভাষ্যকার কে কিং জিন্দেগি বলেন, ‘কসৌটি টু-তে দে..."
9998,সম্ভবত অনুশীলন থেকেই বেল অভিমান উঠেছিল,"তখনই গুঞ্জন উঠেছিল, সম্ভবত অভিমান থেকেই বেল অন...",বেল সম্ভবত অনুশীলন থেকেই এমন অভিমান গড়ে উঠেছিল।


In [ ]:
rougeScore(df, 'text', 'generatedText')

{'rouge-1': {'r': 0.5801948711067572,
  'p': 0.5101481921741152,
  'f': 0.5388538806722936},
 'rouge-2': {'r': 0.21429575237994208,
  'p': 0.18583563548017015,
  'f': 0.19742124630838553},
 'rouge-l': {'r': 0.44992190959049255,
  'p': 0.3985372828769526,
  'f': 0.4197630746005098}}

In [ ]:
bertScore(df, 'text', 'generatedText')

{'precision': 0.8387137514650822,
 'recall': 0.8218448148369789,
 'f1': 0.8298264009594918}

In [ ]:
bleuCorpass(df, 'text', 'generatedText')

/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

{'Bleu 1': 0.48689792873690757,
 'Bleu 2': 0.26374448975121734,
 'Bleu 3': 0.12748623187146652,
 'Bleu 4': 0.057763854849158795}

In [ ]:
wilWerMeteorScore(df, 'text', 'generatedText')

{'METEOR': 0.370606245292168,
 'WER': 0.7386076870924834,
 'WIL': 0.8268261236482287}

### Bangla BERTScore

In [ ]:
import pandas as pd
df =  pd.read_csv('/content/drive/MyDrive/BengaliKey2Text/DataV3/predictedData.csv')
df

,keywords,text,generatedText
0,প্রায়ই প্রতীক হতে একটা পেরিয়ে হয়—পঞ্চাশ মুখোমু...,বলিউডের তিন খানকে প্রায়ই একটা প্রশ্নের মুখোমুখ...,"আজও এর মুখোমুখি হতে হয়—পঞ্চাশ পেরিয়ে গেল, প্..."
1,স্থানে বা কোনো বোঝা অন্য পড়ার অন্য দুটোই কোচিং...,নাকি অন্য উপায়ে বইয়ের বোঝা ও পড়ার বোঝা দুটোই স...,অন্য কোনো স্থানে পড়ার টেবিলে শিশুদের বই ও কম্...
2,আমার আত্মা দেখেই শুকিয়ে,দেখেই আমার আত্মা শুকিয়ে গেল।,আমার আত্মা শুকিয়ে গেছে দেখেই।
3,বড় অবহেলিত দুই হয়েছে নির্বাচন ঐক্যের বক্তৃতায় ...,"উপজেলা নির্বাচন প্রসঙ্গে তিনি বলেন, এটি নির্দল...","তিনি বলেন, এটি দেশের সংবিধানকেও অবহেলিত করে।সভ..."
4,বলেন কথা সেই সময়ের,সেই সময়ের কিছু কথা বলেন।,তিনি সেই সময়ের কথা বলেন।
...,...,...,...
9995,অনুসরণ মুসলিমদের সময় যেসব নিপীড়ন সরকার রাশিয়া ...,ভারতীয় রাষ্ট্রকে লক্ষ্যবস্তু বলে চিহ্নিত করার ...,কাশ্মীরে মুসলিমদের ওপর যেসব নিপীড়ন চালানো হয়...
9996,অভিযোগ করেনি ব্যাপারে পর্যন্ত গতকাল তবে,তবে গতকাল পর্যন্ত এ ব্যাপারে থানায় কেউ অভিযোগ ...,তবে এ ব্যাপারে গতকাল পর্যন্ত কেউ অভিযোগ করেনি।
9997,জিন্দেগি গেছে কিং কসৌটি টুতে দেখা জানা কে ভাষ্...,"জানা গেছে, বলিউডের কিং খানকে ‘কসৌটি জিন্দেগি ক...","ভাষ্যকার কে কিং জিন্দেগি বলেন, ‘কসৌটি টু-তে দে..."
9998,সম্ভবত অনুশীলন থেকেই বেল অভিমান উঠেছিল,"তখনই গুঞ্জন উঠেছিল, সম্ভবত অভিমান থেকেই বেল অন...",বেল সম্ভবত অনুশীলন থেকেই এমন অভিমান গড়ে উঠেছিল।


In [ ]:
cands = df["generatedText"].values.tolist()
refs = df["text"].values.tolist()

In [ ]:
print(type(cands), ' ',len(cands), '\n', type(refs), ' ',len(refs))

<class 'list'>   10000 
 <class 'list'>   10000


In [ ]:
!git clone https://github.com/csebuetnlp/banglaparaphrase.git
%cd /content/banglaparaphrase/BERTScore/utils
!pip install transformers
!pip install git+https://github.com/csebuetnlp/normalizer
!pip install jsonlines

In [ ]:
from score import score
P, R, F1 = score(cands, refs, lang='bn', verbose=True)

F1_mean= F1.mean()

Some weights of the model checkpoint at csebuetnlp/banglabert were not used when initializing ElectraModel: ['discriminator_predictions.dense.bias', 'discriminator_predictions.dense.weight', 'discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense_prediction.weight']
- This IS expected if you are initializing ElectraModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


calculating scores...
computing bert embedding.


  0%|          | 0/311 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/157 [00:00<?, ?it/s]

done in 23.71 seconds, 421.72 sentences/sec


In [ ]:
print(F1_mean)

tensor(0.9136)


### Multilingual ROUGE

In [ ]:
import pandas as pd
df =  pd.read_csv('/content/drive/MyDrive/BengaliKey2Text/DataV3/predictedData.csv')

In [ ]:
!git clone https://github.com/csebuetnlp/xl-sum.git
%cd /content/xl-sum/multilingual_rouge_scoring
!pip3 install -r requirements.txt
!pip3 install --upgrade ./

In [ ]:
from rouge_score import rouge_scorer

rougescorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True, lang="bengali")

def rougeScore2(test_df, columnName1, columnName2):

    newDf = test_df.filter([columnName1, columnName2], axis=1)

    newDf =  newDf.reset_index()

    lenS = newDf.shape[0]

    rscore1 = 0
    rscoreL = 0

    for i in range(lenS):
        score = rougescorer.score(newDf[columnName1][i], newDf[columnName2][i])

        rscore1 = rscore1 + score['rouge1'].fmeasure
        rscoreL = rscoreL + score['rougeL'].fmeasure

    print(rscore1/lenS)
    print(rscoreL/lenS)

    return rscoreL/lenS

In [ ]:
rougeScore2(df, 'text', 'generatedText')

0.6138287328387928
0.4615779101464652


0.4615779101464652